### Merge ISPU Data and Formating

In [25]:
import pandas as pd
import os
import re

# CONFIG
FOLDER_PATH = "data/ISPU"
OUTPUT_PATH = "data/ISPU/ispu_dki_2010_2025.csv"

FILES_WITH_YEAR = {
    "data-indeks-standar-pencemar-udara-(ispu)-di-provinsi-dki-jakarta-komponen-data-2024.csv": 2024,
    "data-indeks-standar-pencemar-udara-(ispu)-di-provinsi-dki-jakarta-komponen-data-2025.csv": 2025,
}

COLUMN_MAPPING = {
    "lokasi_spku": "stasiun",

    "pm_sepuluh": "pm10",
    "pm_10": "pm10",

    "pm_duakomalima": "pm2.5",
    "pm25": "pm2.5",

    "sulfur_dioksida": "so2",
    "nitrogen_dioksida": "no2",
    "karbon_monoksida": "co",
    "ozon": "o3",

    "categori": "kategori",
    "parameter_pencemar_kritis": "critical"
}

REQUIRED_COLUMNS = ["pm10", "pm2.5", "so2", "no2", "co", "o3"]

# HELPERS
def clean_tanggal(x):
    if pd.isna(x):
        return pd.NaT

    try:
        x = float(x)
        if x > 30000:
            return pd.to_datetime(x, unit="D", origin="1899-12-30")
    except:
        pass

    return pd.to_datetime(x, dayfirst=True, errors="coerce")


def clean_stasiun(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).upper().strip()

    match = re.search(r"DKI\s*([1-5])", x)
    if match:
        return f"DKI{match.group(1)}"

    return pd.NA


def process_ispu(df: pd.DataFrame, year: int | None = None) -> pd.DataFrame:
    df = df.copy()

    df.columns = df.columns.str.lower().str.strip()
    df.rename(columns=COLUMN_MAPPING, inplace=True)

    if {"bulan", "tanggal"}.issubset(df.columns) and year is not None:
        df["year"] = year
        df.rename(columns={"bulan": "month", "tanggal": "day"}, inplace=True)

        df["tanggal"] = pd.to_datetime(
            df[["year", "month", "day"]],
            errors="coerce"
        )

        df.drop(columns=["year", "month", "day"], inplace=True)

    if "tanggal" in df.columns:
        df["tanggal"] = (
            df["tanggal"]
            .apply(clean_tanggal)
            .dt.date
        )

    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    if "stasiun" in df.columns:
        df["stasiun"] = df["stasiun"].apply(clean_stasiun)

    return df


# MAIN PIPELINE
dfs = []

for file in os.listdir(FOLDER_PATH):
    if not file.endswith(".csv"):
        continue
    if file == os.path.basename(OUTPUT_PATH):
        continue  

    path = os.path.join(FOLDER_PATH, file)
    df_raw = pd.read_csv(path)

    year = FILES_WITH_YEAR.get(file)  
    df_clean = process_ispu(df_raw, year)
    dfs.append(df_clean)


# concat semua
df_all = pd.concat(dfs, ignore_index=True)

df_all["tanggal"] = pd.to_datetime(df_all["tanggal"], errors="coerce")

df_all = (
    df_all
    .sort_values(["tanggal", "stasiun"])
    .reset_index(drop=True)
)

df_all.to_csv(OUTPUT_PATH, index=False)

print("Selesai Merge ISPU")


C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_14324\2564853698.py:46: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")
C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_14324\2564853698.py:46: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")
C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_14324\2564853698.py:46: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")
C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_14324\2564853698.py:46: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified

Selesai Merge ISPU


C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_14324\2564853698.py:46: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")


### Merge Cuaca

In [26]:
from pathlib import Path

data_cuaca_path = Path("data/cuaca-harian")

dfs = []

for file_path in data_cuaca_path.glob("cuaca-harian-dki[1-5]-*.csv"):
    filename = file_path.stem
    stasiun = filename.split("-")[2].strip().upper()  # DKI1–DKI5

    df = pd.read_csv(file_path)

    # ubah nama kolom time -> tanggal
    df = df.rename(columns={"time": "tanggal"})

    # pastikan tanggal bertipe datetime
    df['tanggal'] = pd.to_datetime(df['tanggal'], errors='coerce')

    df['stasiun'] = stasiun
    dfs.append(df)

if not dfs:
    raise ValueError("Tidak ada file DKI1–DKI5 yang terbaca")

df_cuaca_all = pd.concat(dfs, ignore_index=True)

# sorting
stasiun_order = ['DKI1','DKI2','DKI3','DKI4','DKI5']
df_cuaca_all['stasiun'] = pd.Categorical(
    df_cuaca_all['stasiun'],
    categories=stasiun_order,
    ordered=True
)

df_cuaca_all = (
    df_cuaca_all
    .sort_values(['tanggal', 'stasiun'])
    .reset_index(drop=True)
)

output_path = data_cuaca_path / "cuaca-harian-dki-gabungan.csv"
df_cuaca_all.to_csv(output_path, index=False)

print("Selesai. File tersimpan di:", output_path)


Selesai. File tersimpan di: data\cuaca-harian\cuaca-harian-dki-gabungan.csv


### Perbaikan sorted NDVI

In [27]:
df = pd.read_csv("data/NDVI (vegetation index)/indeks-ndvi-jakarta.csv")  

df["tanggal"] = pd.to_datetime(df["tanggal"])

df = df.sort_values(by=["tanggal", "stasiun_id"])

df = df.reset_index(drop=True)

df = df.rename(columns={"stasiun_id": "stasiun"})

df.to_csv("data/NDVI (vegetation index)/ndvi_sorted.csv", index=False)

### MERGE ISPU, CUACA HARIAN DAN NDVI

In [28]:
BASE_DIR = Path("data")

df_ispu = pd.read_csv(BASE_DIR / "ISPU" / "ispu_dki_2010_2025.csv")
df_cuaca = pd.read_csv(BASE_DIR / "cuaca-harian" / "cuaca-harian-dki-gabungan.csv")
df_ndvi = pd.read_csv(BASE_DIR / "NDVI (vegetation index)" / "ndvi_sorted.csv")

for df in [df_ispu, df_cuaca, df_ndvi]:
    df["tanggal"] = pd.to_datetime(df["tanggal"])

df_ispu = df_ispu.sort_values(["tanggal", "stasiun"])
df_cuaca = df_cuaca.sort_values(["tanggal", "stasiun"])
df_ndvi = df_ndvi.sort_values(["tanggal", "stasiun"])

df_merge = pd.merge(
    df_ispu,
    df_cuaca,
    on=["tanggal", "stasiun"],
    how="inner"
)

df_merge = pd.merge(
    df_merge,
    df_ndvi,
    on=["tanggal", "stasiun"],
    how="left"   
)

df_merge = df_merge.reset_index(drop=True)

kolom = [c for c in df_merge.columns if c != "kategori"] + ["kategori"]
df_merge = df_merge[kolom]

df_merge.to_csv("data/merged_ispu_cuaca_ndvi.csv", index=False)


# EDA (Exploratory Data Analysis)

In [29]:
# Library
import pandas as pd

In [30]:
df = pd.read_csv("data/merged_ispu_cuaca_ndvi.csv")

C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_14324\4199542109.py:1: DtypeWarning: Columns (3,4,5,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/merged_ispu_cuaca_ndvi.csv")


In [32]:
date_col = "tanggal"   
df[date_col] = pd.to_datetime(
    df[date_col],
    errors="coerce",
    infer_datetime_format=True
)

C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_14324\1887891164.py:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[date_col] = pd.to_datetime(


In [33]:
print("Jumlah tanggal invalid:", df[date_col].isna().sum())

Jumlah tanggal invalid: 0


In [34]:
full_range = pd.date_range(
    start=df[date_col].min(),
    end=df[date_col].max(),
    freq="D"   
)

missing_dates = full_range.difference(df[date_col].unique())

print("Jumlah tanggal hilang:", len(missing_dates))

Jumlah tanggal hilang: 115


In [35]:
entity_col = "stasiun"
dup_rows = df.duplicated(subset=[entity_col, date_col]).sum()
print("Jumlah duplikasi (entitas + tanggal):", dup_rows)

# lihat contoh duplikat
df_dups = df[df.duplicated(subset=[entity_col, date_col], keep=False)]
df_dups.head()


Jumlah duplikasi (entitas + tanggal): 215


,periode_data,tanggal,stasiun,pm10,pm2.5,so2,co,o3,no2,max,...,cloud_cover_max (%),cloud_cover_min (%),wind_gusts_10m_mean (km/h),wind_speed_10m_mean (km/h),wind_gusts_10m_min (km/h),wind_speed_10m_min (km/h),surface_pressure_max (hPa),surface_pressure_min (hPa),ndvi,kategori
5147,201609,2016-01-08,DKI2,56,NaN,17,30,66,23,66,...,97,28,15.5,5.1,5.4,1.3,1012.7,1008.9,NaN,SEDANG
5148,201608,2016-01-08,DKI2,60,NaN,18,21,70,11,70,...,97,28,15.5,5.1,5.4,1.3,1012.7,1008.9,NaN,SEDANG
5149,201609,2016-01-08,DKI3,55,NaN,25,14,85,3,85,...,97,28,15.7,5.3,5.0,1.1,1005.4,1001.6,NaN,SEDANG
5150,201608,2016-01-08,DKI3,55,NaN,24,12,100,4,100,...,97,28,15.7,5.3,5.0,1.1,1005.4,1001.6,NaN,SEDANG
5151,201609,2016-01-08,DKI4,61,NaN,21,16,---,11,61,...,97,28,15.5,5.1,5.4,1.3,1009.5,1005.7,NaN,SEDANG


In [36]:
df[date_col].dtype

dtype('<M8[ns]')

In [37]:
def is_sequential(group):
    diffs = group[date_col].diff().dropna()
    return diffs.min(), diffs.max()

if entity_col:
    seq_check = df.groupby(entity_col).apply(is_sequential)
else:
    seq_check = is_sequential(df)

print(seq_check)


stasiun
DKI1    (0 days 00:00:00, 245 days 00:00:00)
DKI2     (0 days 00:00:00, 70 days 00:00:00)
DKI3     (0 days 00:00:00, 86 days 00:00:00)
DKI4    (0 days 00:00:00, 108 days 00:00:00)
DKI5    (0 days 00:00:00, 498 days 00:00:00)
dtype: object


C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_14324\511690531.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  seq_check = df.groupby(entity_col).apply(is_sequential)
